## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
from IPython.display import display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load the Dataset

In [2]:
df = pd.read_excel("hotel_dataset.xlsx")

df.head()

,countyName,cityName,HotelCode,HotelName,HotelRating,Address,Attractions,Description,HotelFacilities,Map
0,Egypt,6th of October City,1029293,Hilton Pyramids Golf Resort,FiveStar,Giza Al Wahat Al Baharia Dreamland6th of Octob...,Distances are displayed to the nearest 0.1 mil...,<p>HeadLine : Near Dreamland Golf Course</p><p...,Number of bars/lounges - 2 Designated smoking ...,29.962074|31.04513
1,Egypt,6th of October City,1051510,Movenpick Hotel & Casino Cairo Media City,FiveStar,El Hay El Motamayez 12451 P.O. Box 39 6Th Of O...,Distances are displayed to the nearest 0.1 mil...,<p>HeadLine : Near Mall Of Egypt</p><p>Locatio...,Volleyball on site Shopping mall on site Walkw...,29.9656|31.02077
2,Egypt,6th of October City,1165011,Novotel Cairo 6th of October,FourStar,Ext. 26Th Of July St. 6th Of OctoberBehind Dar...,Distances are displayed to the nearest 0.1 mil...,<p>HeadLine : Near Mall of Arabia</p><p>Locati...,Gift shops or newsstand Dry cleaning/laundry s...,29.997168|30.969372
3,Egypt,6th of October City,1200831,Swiss Inn Plaza,FourStar,Dreamland Al Wahat Road 6th of October CityCairo,NaN,Featuring panoramic views of the Dreamland Gol...,Security alarm Smoke alarms CCTV in common are...,29.96164|31.04863
4,Egypt,6th of October City,1288665,Swiss Inn Pyramids Golf Resort,FiveStar,Dreamland Dreamland6th of October CityCairo,Distances are displayed to the nearest 0.1 mil...,<p>HeadLine : Near Dreamland Golf Course</p><p...,24-hour front desk Lifeguard on site Golfing n...,29.961758|31.045818


In [3]:
print("Shape before cleaning:", df.shape)

Shape before cleaning: (148369, 10)


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 148369 entries, 0 to 148368
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype
---  ------           --------------   -----
 0   countyName       148369 non-null  str  
 1   cityName         148369 non-null  str  
 2   HotelCode        148369 non-null  int64
 3   HotelName        148369 non-null  str  
 4   HotelRating      148369 non-null  str  
 5   Address          148365 non-null  str  
 6   Attractions      68714 non-null   str  
 7   Description      145914 non-null  str  
 8   HotelFacilities  145229 non-null  str  
 9   Map              148342 non-null  str  
dtypes: int64(1), str(9)
memory usage: 11.3 MB


In [5]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
countyName,148369,4,Spain,81620,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cityName,148369,8005,Gran Canaria,2055,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HotelCode,148369.0,NaN,NaN,NaN,3471307.531135,2128033.971531,1000000.0,1346042.0,1922535.0,5560713.0,6194291.0
HotelName,148369,110593,Lloret De Mar Villa,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HotelRating,148369,6,All,57850,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Address,148365,108992,123 Main St,132,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Attractions,68714,50865,Beach,44,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,145914,113043,HotelDescription#Further information about thi...,44,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HotelFacilities,145229,99550,Hotel,1113,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Map,148342,109739,45.37354|6.5788,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Explore Column Values

In [6]:
print("Countries covered:")
print(df['countyName'].value_counts())
print()
print("Number of unique cities:", df['cityName'].nunique())
print()
print("Hotel rating categories:")
print(df['HotelRating'].value_counts())

Countries covered:
countyName
Spain          81620
France         56247
Switzerland     8288
Egypt           2214
Name: count, dtype: int64

Number of unique cities: 8005

Hotel rating categories:
HotelRating
All          57850
ThreeStar    47576
TwoStar      17446
FourStar     15919
OneStar       7142
FiveStar      2436
Name: count, dtype: int64


## 4. Rename the Misspelled Column

In [7]:
df = df.rename(columns={'countyName': 'countryName'})
df.columns.tolist()

['countryName',
 'cityName',
 'HotelCode',
 'HotelName',
 'HotelRating',
 'Address',
 'Attractions',
 'Description',
 'HotelFacilities',
 'Map']

## 5. Check for Missing Values

In [8]:
missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'missing_count': missing_count,
    'missing_percent': missing_percent
})
missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_percent', ascending=False)

,missing_count,missing_percent
Attractions,79655,53.69
HotelFacilities,3140,2.12
Description,2455,1.65
Map,27,0.02
Address,4,0.00


## 6. Handle Missing Values

In [9]:
df = df.drop(columns=['Attractions'])

before = len(df)
df = df.dropna(subset=['Address', 'Map'])
print(f"Dropped {before - len(df)} rows due to missing Address/Map")

text_fill_map = {
    'Description': 'No description available',
    'HotelFacilities': 'No facilities information available'
}

for col, placeholder in text_fill_map.items():
    df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)
    df[col] = df[col].fillna(placeholder)

# Confirm no missing values remain
df.isnull().sum()

Dropped 31 rows due to missing Address/Map


countryName        0
cityName           0
HotelCode          0
HotelName          0
HotelRating        0
Address            0
Description        0
HotelFacilities    0
Map                0
dtype: int64

## 7. Check for Duplicate Records

In [10]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Rows with duplicated HotelCode:", df.duplicated(subset=['HotelCode']).sum())

Fully duplicated rows: 0
Rows with duplicated HotelCode: 32728


In [11]:
dup_codes = df[df.duplicated(subset=['HotelCode'], keep=False)]['HotelCode'].unique()
example_code = dup_codes[0]
df[df['HotelCode'] == example_code][['countryName', 'cityName', 'HotelCode', 'HotelName', 'HotelRating']]

,countryName,cityName,HotelCode,HotelName,HotelRating
0,Egypt,6th of October City,1029293,Hilton Pyramids Golf Resort,FiveStar
280,Egypt,Cairo,1029293,Hilton Pyramids Golf Resort,FiveStar


## 8. Handle Duplicate Records

In [12]:
before = len(df)
df = df.drop_duplicates(subset=['HotelCode'], keep='first')
print(f"Dropped {before - len(df)} duplicate hotel records")
print("Remaining rows:", len(df))
print("Duplicated HotelCode after cleaning:", df.duplicated(subset=['HotelCode']).sum())

Dropped 32728 duplicate hotel records
Remaining rows: 115610
Duplicated HotelCode after cleaning: 0


## 9. Clean HTML from Text Fields

In [13]:
def clean_html(text):
    if pd.isnull(text):
        return text
    text = re.sub(r'<[^>]+>', ' ', str(text))
    text = text.replace('\\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for col in ['Description', 'HotelFacilities', 'Address']:
    df[col] = df[col].apply(clean_html)

df[['Description', 'HotelFacilities']].head(3)

,Description,HotelFacilities
0,HeadLine : Near Dreamland Golf Course Location...,Number of bars/lounges - 2 Designated smoking ...
1,HeadLine : Near Mall Of Egypt Location : Locat...,Volleyball on site Shopping mall on site Walkw...
2,HeadLine : Near Mall of Arabia Location : Loca...,Gift shops or newsstand Dry cleaning/laundry s...


## 10. Standardize the Hotel Rating

In [14]:
rating_map = {
    'OneStar': 1,
    'TwoStar': 2,
    'ThreeStar': 3,
    'FourStar': 4,
    'FiveStar': 5,
    'All': 0
}

df['HotelRatingNumeric'] = df['HotelRating'].map(rating_map)
df[['HotelRating', 'HotelRatingNumeric']].drop_duplicates()

,HotelRating,HotelRatingNumeric
0,FiveStar,5
2,FourStar,4
6,ThreeStar,3
7,All,0
17,TwoStar,2
82,OneStar,1


## 11. Split the Map Column into Latitude and Longitude

In [15]:
coords = df['Map'].astype(str).str.split('|', expand=True)
df['Latitude'] = pd.to_numeric(coords[0], errors='coerce')
df['Longitude'] = pd.to_numeric(coords[1], errors='coerce')

before = len(df)
df = df.dropna(subset=['Latitude', 'Longitude'])
print(f"Dropped {before - len(df)} rows with invalid coordinates")

df = df.drop(columns=['Map'])
df[['Latitude', 'Longitude']].describe()

Dropped 1 rows with invalid coordinates


,Latitude,Longitude
count,115609.000000,115609.000000
mean,41.781732,0.964122
std,5.118545,7.294329
min,-37.694220,-180.000000
25%,38.911860,-2.894118
50%,42.731050,1.316730
75%,45.532150,4.987400
max,85.051129,177.454345


## 12. Final Data Type Check

In [16]:
df['countryName'] = df['countryName'].astype('category')
df['cityName'] = df['cityName'].astype('category')
df['HotelRating'] = df['HotelRating'].astype('category')

df.dtypes

countryName           category
cityName              category
HotelCode                int64
HotelName                  str
HotelRating           category
Address                    str
Description                str
HotelFacilities            str
HotelRatingNumeric       int64
Latitude               float64
Longitude              float64
dtype: object

## 13. Final Checks

In [17]:
print("Missing values left:\n", df.isnull().sum())
print()
print("Duplicated HotelCode left:", df.duplicated(subset=['HotelCode']).sum())
print("Final shape:", df.shape)
df.head()

Missing values left:
 countryName           0
cityName              0
HotelCode             0
HotelName             0
HotelRating           0
Address               0
Description           0
HotelFacilities       0
HotelRatingNumeric    0
Latitude              0
Longitude             0
dtype: int64

Duplicated HotelCode left: 0
Final shape: (115609, 11)


,countryName,cityName,HotelCode,HotelName,HotelRating,Address,Description,HotelFacilities,HotelRatingNumeric,Latitude,Longitude
0,Egypt,6th of October City,1029293,Hilton Pyramids Golf Resort,FiveStar,Giza Al Wahat Al Baharia Dreamland6th of Octob...,HeadLine : Near Dreamland Golf Course Location...,Number of bars/lounges - 2 Designated smoking ...,5,29.962074,31.045130
1,Egypt,6th of October City,1051510,Movenpick Hotel & Casino Cairo Media City,FiveStar,El Hay El Motamayez 12451 P.O. Box 39 6Th Of O...,HeadLine : Near Mall Of Egypt Location : Locat...,Volleyball on site Shopping mall on site Walkw...,5,29.965600,31.020770
2,Egypt,6th of October City,1165011,Novotel Cairo 6th of October,FourStar,Ext. 26Th Of July St. 6th Of OctoberBehind Dar...,HeadLine : Near Mall of Arabia Location : Loca...,Gift shops or newsstand Dry cleaning/laundry s...,4,29.997168,30.969372
3,Egypt,6th of October City,1200831,Swiss Inn Plaza,FourStar,Dreamland Al Wahat Road 6th of October CityCairo,Featuring panoramic views of the Dreamland Gol...,Security alarm Smoke alarms CCTV in common are...,4,29.961640,31.048630
4,Egypt,6th of October City,1288665,Swiss Inn Pyramids Golf Resort,FiveStar,Dreamland Dreamland6th of October CityCairo,HeadLine : Near Dreamland Golf Course Location...,24-hour front desk Lifeguard on site Golfing n...,5,29.961758,31.045818


## 14. Feature Engineering: Target Variable & Features

In [18]:
import numpy as np

np.random.seed(42)

df['FacilitiesCount'] = df['HotelFacilities'].apply(lambda x: len(str(x).split(',')) if pd.notnull(x) else 0)
df['DescriptionLength'] = df['Description'].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)

df['HotelValue'] = (
    50 
    + (df['HotelRatingNumeric'] * 30) 
    + (df['FacilitiesCount'] * 5) 
    + (df['DescriptionLength'] * 0.1)
    + np.random.normal(0, 20, size=len(df))
)
df['HotelValue'] = df['HotelValue'].clip(lower=20)

df[['HotelRatingNumeric', 'FacilitiesCount', 'DescriptionLength', 'HotelValue']].head()

,HotelRatingNumeric,FacilitiesCount,DescriptionLength,HotelValue
0,5,1,343,249.234283
1,5,1,310,233.234714
2,4,1,333,221.253771
3,4,1,239,229.360597
4,5,1,428,243.116933


## 15. Regression Pipeline & Model Evaluation

In [19]:
features = ['countryName', 'HotelRatingNumeric', 'Latitude', 'Longitude', 'FacilitiesCount', 'DescriptionLength']
target = 'HotelValue'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
categorical_features = ['countryName']
numerical_features = ['HotelRatingNumeric', 'Latitude', 'Longitude', 'FacilitiesCount', 'DescriptionLength']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

models = {
    'Baseline': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('model', model)])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2
    })

# Present results clearly in a comparison table
results_df = pd.DataFrame(results)
display(results_df.round(3))

,Model,MAE,RMSE,R²
0,Baseline,46.576,53.204,-0.000
1,Linear Regression,15.975,20.008,0.859
2,Random Forest Regressor,17.119,21.452,0.837
